# Squat-Only YOLO Pose Extraction (Google Colab)

This notebook keeps the local pipeline unchanged and provides a separate Colab workflow for running squat-only pose extraction with a GPU.

## 1. Select a GPU Runtime

In Colab:
- `Runtime` -> `Change runtime type`
- `Hardware accelerator` -> `GPU`

In [1]:
import os

# Choose ONE of the options below.
# Option A: clone from GitHub
REPO_URL = "https://github.com/lindaperez/CV_Image_pose_detection.git"
WORKDIR = "/content/CV_Image_pose_detection"

# Option B: use Google Drive
# WORKDIR = "/content/drive/MyDrive/personal-git/Final_Project/CV_Image_pose_detection""

## 2. Get the Repo into Colab

In [2]:
# Run this cell if you want to clone from GitHub.
!rm -r /content/CV_Image_pose_detection
!git clone $REPO_URL /content/CV_Image_pose_detection

rm: cannot remove '/content/CV_Image_pose_detection': No such file or directory
Cloning into '/content/CV_Image_pose_detection'...
remote: Enumerating objects: 203, done.
remote: Total 203 (delta 0), reused 0 (delta 0), pack-reused 203 (from 1)
Receiving objects: 100% (203/203), 440.90 MiB | 26.95 MiB/s, done.
Resolving deltas: 100% (65/65), done.
Updating files: 100% (139/139), done.


In [3]:
%cd $WORKDIR
!pwd
!ls /content/CV_Image_pose_detection/artifacts/3_Modeling

/content/CV_Image_pose_detection
/content/CV_Image_pose_detection
build_pose_feature_index.py  Squat_Pose_Extraction_Colab.ipynb
COLAB_SQUAT_POSE.md	     training_outputs
Model_Training_01.ipynb      yolo11n-pose.pt
pose_feature_extraction.py   YOLO_POSE_STAGE.md
__pycache__


In [4]:
cd /content/

/content


## 3. Install Pose Dependencies

In [5]:
!python3 -m pip install -r CV_Image_pose_detection/requirements-pose.txt

## 4. Confirm GPU

In [6]:
import torch

print("cuda_available =", torch.cuda.is_available())
print("device_count =", torch.cuda.device_count())
print("device_name =", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

cuda_available = True
device_count = 1
device_name = Tesla T4


In [7]:
#connect with drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 5. Build a Squat-Only Index

In [8]:
!python3 CV_Image_pose_detection/artifacts/3_Modeling/build_pose_feature_index.py \
  --exercise squat \
  --output-csv CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_feature_index_squat.csv

Wrote 118 rows for squat to /content/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_feature_index_squat.csv


In [11]:
ls -l CV_Image_pose_detection/Data/LLSP/annotation_cleaned/

total 564
drwxr-xr-x 2 root root   4096 Mar  8 18:43 _archived_test_outputs_20260304_114631/
-rw-r--r-- 1 root root    178 Mar  8 18:43 class_weights_train.csv
-rw-r--r-- 1 root root   3654 Mar  8 18:43 decisions_manifest.json
-rw-r--r-- 1 root root   1194 Mar  8 18:44 pose_extraction_report.csv
-rw-r--r-- 1 root root    635 Mar  8 18:44 pose_extraction_summary.json
-rw-r--r-- 1 root root 170393 Mar  8 18:43 pose_feature_index.csv
-rw-r--r-- 1 root root  13950 Mar  8 18:44 pose_feature_index_squat.csv
drwxr-xr-x 2 root root   4096 Mar  8 18:43 pose_features/
-rw-r--r-- 1 root root 281824 Mar  8 18:43 train_cleaned.csv
-rw-r--r-- 1 root root  26008 Mar  8 18:43 train_sample_weights.csv
-rw-r--r-- 1 root root  51740 Mar  8 18:43 valid_cleaned.csv


In [12]:
!head -n 5 CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_feature_index_squat.csv

name,feature_path,type,split,count
test2340.mp4,/content/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_features/test2340.npy,squat,train,4.0
stu4_66.mp4,/content/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_features/stu4_66.npy,squat,train,27.0
stu9_63.mp4,/content/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_features/stu9_63.npy,squat,train,20.0
stu1_68.mp4,/content/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_features/stu1_68.npy,squat,train,20.0


In [26]:
# Check on number of videos in the directory

from pathlib import Path

def count_mp4_videos(directory: str, recursive: bool = True) -> int:
    """
    Count .mp4 video files in a directory.

    Args:
        directory: Path to the folder.
        recursive: If True, also count videos in subfolders.

    Returns:
        Number of .mp4 files found.
    """
    path = Path(directory)

    if not path.exists():
        raise FileNotFoundError(f"Directory does not exist: {directory}")

    if not path.is_dir():
        raise NotADirectoryError(f"Not a directory: {directory}")

    pattern = "**/*.mp4" if recursive else "*.mp4"
    return sum(1 for _ in path.glob(pattern))


drive_dir = "/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/video/test"
total_videos = count_mp4_videos(drive_dir)
print(f"Total .mp4 videos: {total_videos}")



Total .mp4 videos: 22


## 6. Smoke Test on 5 Squat Videos

In [27]:
  !python3 /content/CV_Image_pose_detection/artifacts/3_Modeling/pose_feature_extraction.py \
  --index-csv /content/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_feature_index_squat.csv \
  --video-dir /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/video/ \
  --model /content/CV_Image_pose_detection/artifacts/3_Modeling/yolo11n-pose.pt \
  --report-path /content/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_extraction_report.csv \
  --summary-path /content/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_extraction_summary.json \
  --device cuda:0 \
  --max-videos 5 \
  --overwrite

Indexed 1041 videos under /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/video
Processing 5 videos...
[5/5] ok: stu6_62.mp4 | frames=749 used=749 shape=(749, 51)

Done.
ok=5, skipped_exists=0, failed=0
report: /content/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_extraction_report.csv
summary: /content/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_extraction_summary.json


## 7. Run the Full Squat Set

Remove `--overwrite` if you want to resume and skip already-finished files.

In [29]:
!python3 CV_Image_pose_detection/artifacts/3_Modeling/pose_feature_extraction.py \
  --index-csv CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_feature_index_squat.csv \
  --video-dir /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/video \
  --device cuda:0

Indexed 1041 videos under /content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/video
Processing 118 videos...
[25/118] ok: stu5_66.mp4 | frames=990 used=990 shape=(990, 51)
[50/118] ok: stu7_69.mp4 | frames=1109 used=1109 shape=(1109, 51)
[75/118] ok: stu1_67.mp4 | frames=1079 used=1079 shape=(1079, 51)
[100/118] ok: stu9_66.mp4 | frames=1800 used=1800 shape=(1800, 51)
[118/118] ok: stu10_69.mp4 | frames=1920 used=1920 shape=(1920, 51)

Done.
ok=113, skipped_exists=5, failed=0
report: /content/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_extraction_report.csv
summary: /content/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_extraction_summary.json


## 8. Inspect Outputs

In [30]:
import json
from pathlib import Path

summary_path = Path("CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_extraction_summary.json")
print(json.loads(summary_path.read_text()))

{'total_rows': 118, 'ok': 113, 'skipped_exists': 5, 'failed': 0, 'ok_with_zero_pose_frames': 0, 'args': {'index_csv': '/content/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_feature_index_squat.csv', 'discover_from_videos': False, 'video_dir': '/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection/Data/LLSP/video', 'feature_dir': '/content/CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_features', 'write_index_csv': None, 'model': '/content/yolo11n-pose.pt', 'conf': 0.25, 'imgsz': 640, 'device': 'cuda:0', 'overwrite': False, 'max_videos': 0}}


In [31]:
import pandas as pd

report = pd.read_csv("CV_Image_pose_detection/Data/LLSP/annotation_cleaned/pose_extraction_report.csv")
report.head()

,name,video_path,feature_path,status,frames_total,frames_used,feat_dim,message
0,test2340.mp4,/content/drive/MyDrive/FinalProjectCV/CV_Image...,/content/CV_Image_pose_detection/Data/LLSP/ann...,skipped_exists,0,0,0,feature file already exists
1,stu4_66.mp4,/content/drive/MyDrive/FinalProjectCV/CV_Image...,/content/CV_Image_pose_detection/Data/LLSP/ann...,skipped_exists,0,0,0,feature file already exists
2,stu9_63.mp4,/content/drive/MyDrive/FinalProjectCV/CV_Image...,/content/CV_Image_pose_detection/Data/LLSP/ann...,skipped_exists,0,0,0,feature file already exists
3,stu1_68.mp4,/content/drive/MyDrive/FinalProjectCV/CV_Image...,/content/CV_Image_pose_detection/Data/LLSP/ann...,skipped_exists,0,0,0,feature file already exists
4,stu6_62.mp4,/content/drive/MyDrive/FinalProjectCV/CV_Image...,/content/CV_Image_pose_detection/Data/LLSP/ann...,skipped_exists,0,0,0,feature file already exists


# What it means:

total_rows: 118
The squat index contained 118 videos.

ok: 113

113 videos were processed and pose features were written in this run.

skipped_exists: 5

5 feature files already existed, so the script skipped them.

failed: 0

No missing videos and no runtime failures.

ok_with_zero_pose_frames: 0

YOLO found at least some pose frames in every successful video.



* 118 squat videos
* -> YOLO pose extraction completed
* -> pose feature files available
* -> ready for post-processing / feature engineering
'''